# Demo 3 --- Why GMS is not just retrieval

The default in demo 1 quoted a fee figure it read back from a retrieved chunk, with nothing checking that the figure was exact or that the policy applied. This demo puts one query through two retrievers: the ordinary dense retriever and the GMS/GEODE triple-mediated retriever. The difference is not the answer text --- it is that one is grounded and can abstain, and the other cannot.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

query = 'What is the overdraft fee and can it be reversed?'
print('query:', query)

## Ordinary retrieval: it always answers

`DenseRagRetriever` embeds the policy document, pulls the nearest chunks and lets the model answer from them. There is no calibrated gate and no abstention: it returns an answer for any query, and `verified` is always `False`. It cannot tell the caller whether the retrieved text actually grounds the claim.

In [ ]:
from agentlab.capstone.dense_rag import DenseRagRetriever

dense = DenseRagRetriever()
d = dense.search(query)[0]
print('decision :', d['decision'])
print('verified :', d['verified'])
print('answer   :', d['answer'][:280])

## GMS/GEODE retrieval: grounded, verified, or abstains

`PolicyRagRetriever` parses the query to policy entities, retrieves the governing triples from the self-corrected GMS, synthesizes from those and verifies the answer against them. When the query does not bind to a governing policy it returns nothing --- a grounded abstention --- rather than an unverified paraphrase.

In [ ]:
from agentlab.capstone.policy_rag import get_default_retriever

gms = get_default_retriever()
extraction = gms.extract(query)
results = gms.search(query, extraction=extraction)
if results:
    g = results[0]
    print('decision :', g['decision'])
    print('verified :', g['verified'])
    print('policy   :', g.get('id'))
    print('answer   :', g['answer'][:280])
else:
    print('decision : abstain (no governing policy bound) -- returns no answer')

## Abstention is the tell

A query with no governing policy is where the two part ways. The dense retriever still returns its nearest-chunk answer; the GMS retriever abstains. Running an off-policy query through both shows it.

In [ ]:
off = 'What is the interest rate on a 30-year mortgage refinance?'
dd = dense.search(off)[0]
print('dense  ->', dd['decision'], '| verified', dd['verified'], '|', dd['answer'][:120])
gg = gms.search(off, extraction=gms.extract(off))
print('GMS    ->', 'abstain (no answer)' if not gg else (gg[0]['decision'] + ' verified=' + str(gg[0]['verified'])))

Ordinary retrieval returns a fluent answer to any query and cannot say when it is unsupported. GMS retrieval grounds the answer in the policy triples it retrieved, reports whether it verified, and abstains when nothing governs the query. That is the property a regulated decision needs, and it is not something a nearest-neighbor lookup provides.